# Random Forest (Categories): Athleticism / Technique / Intangibles

Instead of raw TF-IDF over all words, each player is scored on three interpretable
dimensions derived from their scouting report:

| Category | What it captures |
|---|---|
| **Athleticism** | Physical traits & movement (speed, burst, fluidity, frame) |
| **Technique** | Position-specific craft & football skill (footwork, leverage, route, vision) |
| **Intangibles** | Mental attributes & character (motor, competitiveness, coachability) |

Each score = proportion of report tokens matching that category's word list.
The three scores become the RF features.

## 1. Imports

In [ ]:
import re

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict
from sklearn.metrics import (
    classification_report,
    average_precision_score,
    PrecisionRecallDisplay,
    ConfusionMatrixDisplay,
)

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

RANDOM_STATE = 42

## 2. Controls

In [ ]:
# ── Controls ──────────────────────────────────────────────────────────────────
# Draft year range (inclusive)
YEAR_MIN = 2014
YEAR_MAX = 2021

# Grade range (inclusive). Set to None to disable the bound.
GRADE_MIN = None
GRADE_MAX = None

# Position filter — choose the aggregation level and optionally restrict to
# specific values at that level (set POS_FILTER = None to include all).
#   "position"  → raw position codes   e.g. ["QB"], ["WR", "TE"]
#   "Pos_Group" → position groups      e.g. ["QB"], ["OL"], ["DB"]
#   "Group"     → broad groups         e.g. ["QB"], ["SKILL O"], ["BIG D"]
POS_LEVEL  = "position"
POS_FILTER = None

# ── Category word lists ───────────────────────────────────────────────────────
# Words are matched against NLTK-lemmatized tokens, so use base forms.
# e.g. "explosive" matches "explosiveness", "run" matches "runner"/"running".
ATHLETICISM = {
    "speed", "fast", "explosive", "burst", "athletic", "agile", "quick", "powerful",
    "strong", "physical", "lean", "rangy", "mobile", "fluid", "flexible", "dynamic",
    "sudden", "twitchy", "acceleration", "lateral", "springy", "long", "frame",
    "twitch", "loose", "smooth", "quick", "compact", "thick", "muscular",
}

TECHNIQUE = {
    "route", "footwork", "hand", "technique", "leverage", "fundamental", "awareness",
    "vision", "coverage", "block", "release", "mechanic", "discipline", "assignment",
    "anticipation", "recognition", "balance", "pad", "anchor", "timing", "instinct",
    "scheme", "positioning", "finish", "tackle", "pass", "rush", "pursuit",
}

INTANGIBLES = {
    "competitive", "tough", "motor", "effort", "smart", "leadership", "character",
    "coachable", "passionate", "disciplined", "accountable", "mature",
    "intensity", "desire", "relentless", "focused", "driven", "prepare",
    "football", "intelligent", "instinctive", "awareness", "diligent",
}
# ──────────────────────────────────────────────────────────────────────────────

## 3. Load & Filter Data

In [ ]:
df = pd.read_csv("../data/processed/draft_enriched_with_contracts.csv")

TARGET = "made_it_contract"

# ── Apply controls ─────────────────────────────────────────────────────────────
df = df[(df["year"] >= YEAR_MIN) & (df["year"] <= YEAR_MAX)].copy()
if GRADE_MIN is not None:
    df = df[df["grade"] >= GRADE_MIN]
if GRADE_MAX is not None:
    df = df[df["grade"] <= GRADE_MAX]
if POS_FILTER is not None:
    df = df[df[POS_LEVEL].isin(POS_FILTER)]
df = df.copy()

pos_desc = f"{POS_LEVEL}={POS_FILTER}" if POS_FILTER else "all positions"
print(f"Year: {YEAR_MIN}–{YEAR_MAX}  |  Grade: {GRADE_MIN or '—'}–{GRADE_MAX or '—'}  |  {pos_desc}")
print(f"Rows after filters: {len(df)}")

df = df.dropna(subset=[TARGET]).copy()
df[TARGET] = df[TARGET].astype(int)

df['combined_text'] = (
    df['overview'].fillna('') + ' ' +
    df['strengths'].fillna('') + ' ' +
    df['weaknesses'].fillna('')
).str.strip()

df = df[df['combined_text'] != ''].copy().reset_index(drop=True)

print(f"Rows with text + target: {len(df)}")
print(f"\nTarget distribution:")
print(df[TARGET].value_counts())
print(f"\nPositive rate: {df[TARGET].mean():.1%}")

## 4. Preprocess & Score

Each token is lemmatized so dictionary base forms match inflected variants
(e.g. `"explosive"` matches `"explosiveness"`, `"route"` matches `"routes"`).
Score = matched tokens / total tokens — a proportion so report length doesn't dominate.

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def tokenize(text: str) -> list[str]:
    if not isinstance(text, str) or not text.strip():
        return []
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    return [
        lemmatizer.lemmatize(w)
        for w in text.split()
        if w not in stop_words and len(w) > 1
    ]

def category_score(tokens: list[str], word_set: set) -> float:
    if not tokens:
        return 0.0
    return sum(1 for t in tokens if t in word_set) / len(tokens)

df['_tokens'] = df['combined_text'].apply(tokenize)

df['athleticism_score']  = df['_tokens'].apply(lambda t: category_score(t, ATHLETICISM))
df['technique_score']    = df['_tokens'].apply(lambda t: category_score(t, TECHNIQUE))
df['intangibles_score']  = df['_tokens'].apply(lambda t: category_score(t, INTANGIBLES))

score_cols = ['athleticism_score', 'technique_score', 'intangibles_score']
print("Category score summary (proportion of report tokens):")
print(df[score_cols].describe().round(4))
print()
print("Mean scores by outcome:")
print(df.groupby(TARGET)[score_cols].mean().round(4))

## 5. Random Forest on Category Scores

In [ ]:
X = df[score_cols].values
y = df[TARGET].values

clf = RandomForestClassifier(
    n_estimators=500,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(clf, X, y, cv=cv, scoring='average_precision')
print(f"5-fold CV PR-AUC: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")
print(f"Fold scores: {cv_scores.round(4)}")
print(f"Random baseline: {y.mean():.4f}")

## 6. Evaluation

In [ ]:
y_pred_proba = cross_val_predict(clf, X, y, cv=cv, method='predict_proba')[:, 1]
y_pred = (y_pred_proba >= 0.5).astype(int)

baseline = y.mean()
pr_auc = average_precision_score(y, y_pred_proba)

print(classification_report(y, y_pred, target_names=['did not make it', 'made it']))
print(f"PR-AUC (Average Precision): {pr_auc:.4f}  |  Random baseline: {baseline:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ConfusionMatrixDisplay.from_predictions(
    y, y_pred,
    display_labels=['Did Not Make It', 'Made It'],
    ax=axes[0], colorbar=False,
)
axes[0].set_title('Confusion Matrix')

PrecisionRecallDisplay.from_predictions(
    y, y_pred_proba, ax=axes[1], name='RF (Categories)'
)
axes[1].axhline(y=baseline, color='k', linestyle='--', label=f'Random baseline ({baseline:.2f})')
axes[1].set_title('Precision-Recall Curve (Made It)')
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Feature Importance

In [ ]:
clf.fit(X, y)
importances = clf.feature_importances_
labels = ["Athleticism", "Technique", "Intangibles"]

order = np.argsort(importances)
fig, ax = plt.subplots(figsize=(7, 3))
ax.barh([labels[i] for i in order], importances[order], color='steelblue')
ax.set_xlabel('Mean Decrease in Impurity')
ax.set_title('Feature Importance: Category Scores')
plt.tight_layout()
plt.show()

## 8. Score Distributions by Outcome

Do players who made it have systematically higher scores in any category?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

outcome_labels = {0: 'Did Not Make It', 1: 'Made It'}
colors = {0: '#d73027', 1: '#4575b4'}

for ax, col, label in zip(axes, score_cols, labels):
    data_by_outcome = [
        df.loc[df[TARGET] == outcome, col].values
        for outcome in [0, 1]
    ]
    vp = ax.violinplot(data_by_outcome, positions=[0, 1], showmedians=True)
    for i, body in enumerate(vp['bodies']):
        body.set_facecolor(colors[i])
        body.set_alpha(0.6)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Did Not Make It', 'Made It'], fontsize=9)
    ax.set_title(label)
    ax.set_ylabel('Score (proportion of tokens)')

plt.suptitle('Category Score Distributions by Outcome', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()